In [70]:
import sys
sys.path.append('..')

In [71]:
import torch
import torch.nn as N
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset,DataLoader
from src.utils.config import preprocessed_train_data,preprocessed_test_data
from src.utils.config import cat_cl
from models.model import ChurnMLP
import json


Loading data from data mkdir

In [72]:
train_data=torch.load(preprocessed_train_data)
test_data=torch.load(preprocessed_test_data)


Get categorical(embeddings) , numerical features and targets from train & test

In [73]:
X_train_num, X_train_cat =train_data['X_num'],train_data['X_cat']
X_test_num, X_test_cat =test_data['X_num'],test_data['X_cat']

traintarg, testtarg = train_data['target'],test_data['target']

Initialize DataLoader for fututre Batches and dictionary with categories and their classes

In [74]:
train_dataset=TensorDataset(X_train_num,X_train_cat,traintarg)
X_train=DataLoader(train_dataset,batch_size=64,shuffle=True)

In [75]:
test_dataset=TensorDataset(X_test_num,X_test_cat,testtarg)
X_test=DataLoader(test_dataset,batch_size=256,shuffle=False)

In [76]:
with open(cat_cl,'r') as file:
    cat_classes=json.load(file)

Initialize model, pos_weight and optimizer (I'll choose Adam)

In [77]:
churn_predictor=ChurnMLP(num_lay_sz=X_train_num.shape[-1],cat_classes=cat_classes,emb_dim=8)

In [83]:
optimizer=optim.Adam(churn_predictor.parameters(),lr=5e-4)

In [79]:
pos_weight=len(train_data['target'][train_data['target']==0])/len(train_data['target'][train_data['target']==1])
pos_weight

2.768561872909699

Initalize BCEWithLogitsLoss

In [82]:
criterion=N.BCEWithLogitsLoss(pos_weight=None)

Train epochs 

In [84]:
for epoch in range(30):


    churn_predictor.train()
    total_train_loss= 0
    for x_num,x_cat,targ in X_train:


        optimizer.zero_grad()

        logits=churn_predictor(x_num,x_cat)
        batch_loss=criterion(logits,targ)
        batch_loss.backward()
        optimizer.step()

        total_train_loss += batch_loss.item() * targ.size(0) 

    avg_train_loss=total_train_loss/len(train_dataset)


    churn_predictor.eval()
    correct=0
    total=0

    with torch.no_grad():
        for x_num,x_cat,targ in X_test:
            logits=churn_predictor(x_num,x_cat)
            probs=torch.sigmoid(logits)
            preds=(probs>0.35).float()

            correct+= (preds==targ).sum().item()
            total+=targ.size(0)

    print(f"Epoch: {epoch+1}| Train_Avg_loss: {avg_train_loss} | Test_Accuracy: {correct/total}")

Epoch: 1| Train_Avg_loss: 0.3961435425332306 | Test_Accuracy: 0.7601135557132718
Epoch: 2| Train_Avg_loss: 0.3842216169195138 | Test_Accuracy: 0.7679205110007097
Epoch: 3| Train_Avg_loss: 0.38299084336020484 | Test_Accuracy: 0.7672107877927609
Epoch: 4| Train_Avg_loss: 0.3842696272548226 | Test_Accuracy: 0.7537260468417317
Epoch: 5| Train_Avg_loss: 0.3813483825930053 | Test_Accuracy: 0.7572746628814763
Epoch: 6| Train_Avg_loss: 0.3809227986584583 | Test_Accuracy: 0.7771469127040455
Epoch: 7| Train_Avg_loss: 0.3769817586524037 | Test_Accuracy: 0.7735982966643009
Epoch: 8| Train_Avg_loss: 0.3779233818490919 | Test_Accuracy: 0.7665010645848119
Epoch: 9| Train_Avg_loss: 0.373675639700458 | Test_Accuracy: 0.7714691270404542
Epoch: 10| Train_Avg_loss: 0.3767846261075426 | Test_Accuracy: 0.7693399574166075
Epoch: 11| Train_Avg_loss: 0.3753424408717658 | Test_Accuracy: 0.765791341376863
Epoch: 12| Train_Avg_loss: 0.37209426624621744 | Test_Accuracy: 0.7764371894960965
Epoch: 13| Train_Avg_loss